# For the manual version

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [2]:
X = pd.read_csv("../data/customer_features_preprocessed.csv")

In [3]:
X.head()

,total_orders,total_spent,average_order_value,recency_days
0,-0.162369,5.898973,6.011875,0.618548
1,9.166697,5.458187,4.192548,-1.481696
2,4.502164,5.171925,4.408634,1.798250
3,-0.162369,5.122689,5.224472,-1.265149
4,-0.162369,5.062756,5.163681,2.109877


In [4]:
X.shape

(96095, 4)

# a small real sample


In [35]:
sample = X.head(10).to_numpy()

In [37]:
# k = 2 for learning:
centroid_1 = sample[0].copy()
centroid_2 = sample[1].copy()


In [38]:
# Create the  clusters
cluster_1 = []
cluster_2 = []

In [39]:
# Assign every real customer in the sample to the nearest centroid
for customer in sample:
    distance_1 = np.linalg.norm(customer - centroid_1)
    distance_2 = np.linalg.norm(customer - centroid_2)

    if distance_1 < distance_2:
        cluster_1.append(customer)
    else:
        cluster_2.append(customer)

In [40]:
# convert to arrays
cluster_1 = np.array(cluster_1)
cluster_2 = np.array(cluster_2)

In [41]:
#calculate the new centroids
new_centroid_1 = np.mean(cluster_1, axis=0)
new_centroid_2 = np.mean(cluster_2, axis=0)

print("Old centroid 1:", centroid_1)
print("New centroid 1:", new_centroid_1)

print("Old centroid 2:", centroid_2)
print("New centroid 2:", new_centroid_2)

Old centroid 1: [-0.16236915  5.89897255  6.01187469  0.61854817]
New centroid 1: [0.35591231 5.0511787  5.05573963 0.45321361]
Old centroid 2: [ 9.16669721  5.45818715  4.1925484  -1.48169551]
New centroid 2: [ 9.16669721  5.45818715  4.1925484  -1.48169551]


In [42]:
cluster_1

array([[-0.16236915,  5.89897255,  6.01187469,  0.61854817],
       [ 4.50216403,  5.17192483,  4.40863395,  1.79824997],
       [-0.16236915,  5.12268875,  5.22447163, -1.26514947],
       [-0.16236915,  5.06275633,  5.16368075,  2.10987708],
       [-0.16236915,  5.06149387,  5.16240021, -1.33124267],
       [-0.16236915,  5.02620442,  5.12660528,  1.45215249],
       [-0.16236915,  4.9020478 ,  5.00067028,  0.25469297],
       [-0.16236915,  4.61306077,  4.70754388, -1.24486445],
       [-0.16236915,  4.60145902,  4.69577596,  1.68665838]])

In [43]:
cluster_2

array([[ 9.16669721,  5.45818715,  4.1925484 , -1.48169551]])

# the entire dataset

In [79]:
X_array = X.to_numpy()

In [80]:
K = 3
np.random.seed(42)

random_indices = np.random.choice(
    len(X_array),
    size=K,
    replace=False
)

In [81]:
#create initialize 3 centroids from actual customers
centroids = X_array[random_indices].copy()
centroids

array([[-0.16236915,  0.91022544,  0.95167022,  0.2074139 ],
       [-0.16236915, -0.34134251, -0.31782482,  0.09070578],
       [-0.16236915,  0.0394681 ,  0.06844041, -0.01144849]])

In [82]:
# assigns all customers to their closest centroid
def assign_clusters(X, centroids):
    labels = []

    for customer in X:
        distances = []

        for centroid in centroids:
            distance = np.linalg.norm(customer - centroid)
            distances.append(distance)

        nearest_cluster = np.argmin(distances)

        labels.append(nearest_cluster)

    return np.array(labels)

In [83]:
# recalculate the centroids
def update_centroids(X, labels, K):
    new_centroids = []

    for cluster_id in range(K):

        cluster_points = X[labels == cluster_id]

        new_centroid = np.mean(
            cluster_points,
            axis=0
        )

        new_centroids.append(new_centroid)

    return np.array(new_centroids)

In [84]:
# stop condation
def has_converged(old_centroids, new_centroids, tolerance=1e-4):

    movement = np.linalg.norm(
        new_centroids - old_centroids
    )

    return movement < tolerance

In [85]:
def kmeans_from_scratch(X, K, max_iterations=100, tolerance=1e-4):

    np.random.seed(42)

    random_indices = np.random.choice(
        len(X),
        size=K,
        replace=False
    )

    centroids = X[random_indices].copy()

    for iteration in range(max_iterations):

        labels = assign_clusters(X, centroids)

        new_centroids = update_centroids(
            X,
            labels,
            K
        )

        if has_converged(
            centroids,
            new_centroids,
            tolerance
        ):
            print(f"Converged after {iteration + 1} iterations.")
            centroids = new_centroids
            break

        centroids = new_centroids

    return labels, centroids

In [86]:
labels, centroids = kmeans_from_scratch(
    X_array,
    K=3
)

Converged after 69 iterations.


In [87]:
centroids

array([[ 0.20317374,  1.05547687,  1.03579511, -0.22716278],
       [-0.08949701, -0.33265626, -0.32220779,  1.17419061],
       [-0.11476043, -0.6882484 , -0.67836951, -0.61936229]])

In [88]:
np.bincount(labels)

array([32616, 26052, 37427])

In [109]:
# calculate inertia /WCSS -> to know how compact those clusters are. 
def calculate_inertia(X, labels, centroids):
    inertia = 0 
    for i in range(len(X)):
        customer = X[i]

        cluster_id = labels[i]
        
        centroid = centroids[cluster_id]

        squared_distance = np.sum((customer - centroid)**2)

        inertia += squared_distance

    return inertia

In [110]:
inertia = calculate_inertia(X_array, labels, centroids)
print("WCSS / inertia:", inertia)

WCSS / inertia: 218505.4977288083


# Attach clusters to the original data

In [114]:
customer_data = pd.read_csv("../data/cutomer_features.csv")
customer_data.head()

,customer_unique_id,total_orders,total_spent,average_order_value,first_purchase,last_purchase,recency_days
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.080,2017-09-29 15:24:52,2017-09-29 15:24:52,383.087106
1,46450c74a0d8c5ca9395da1daac6c120,3,9553.02,3184.340,2018-07-24 20:41:01,2018-08-17 20:06:36,60.891458
2,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.815,2017-04-01 15:58:40,2017-04-01 15:58:41,564.063623
3,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.880,2018-07-15 14:49:44,2018-07-15 14:49:44,94.111505
4,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.310,2017-02-12 20:37:36,2017-02-12 20:37:36,611.869931


In [115]:
print(customer_data.shape)
print(len(labels))

(96095, 7)
96095


In [116]:
customer_data['cluster'] = labels

In [117]:
customer_data.head()

,customer_unique_id,total_orders,total_spent,average_order_value,first_purchase,last_purchase,recency_days,cluster
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.080,2017-09-29 15:24:52,2017-09-29 15:24:52,383.087106,0
1,46450c74a0d8c5ca9395da1daac6c120,3,9553.02,3184.340,2018-07-24 20:41:01,2018-08-17 20:06:36,60.891458,0
2,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.815,2017-04-01 15:58:40,2017-04-01 15:58:41,564.063623,0
3,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.880,2018-07-15 14:49:44,2018-07-15 14:49:44,94.111505,0
4,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.310,2017-02-12 20:37:36,2017-02-12 20:37:36,611.869931,0


In [120]:
cluster_summary = customer_data.groupby("cluster")[
    [
    "total_orders",
    "total_spent",
    "average_order_value",
    "recency_days"
    ]
].mean()
cluster_summary

,total_orders,total_spent,average_order_value,recency_days
cluster,,,,
0,1.078366,331.836772,317.898879,253.347691
1,1.015623,97.108563,96.119512,468.327489
2,1.010207,70.959780,70.462772,193.180874
